# 🎄 Santa 2026 | Best Public Score, a Gift per Run

**Score of this version: 130,193 instructions.**

**Use it:** Copy & Edit → set `WRITE_SUBMISSION = True` in the first code cell → Run All. About 30 seconds, every one of
the 1,000 programs is replayed before `submission.csv` is written.

Like an advent calendar: every version opens **one new door**, and behind it is **one gift**, a shorter program for
a single puzzle, found by re-running a public optimizer and checked under the competition rules.

* **Input:** only the dataset [santa-2026-elf-programs-and-experiments](https://www.kaggle.com/datasets/asteriospanoras/santa-2026-elf-programs-and-experiments). Its `best_programs.csv`
  is the base: for every puzzle the shortest public program that replays correctly, with its author in the `source`
  column (full credits and history are in the dataset).
* **This version's gift** is in the `CHANGE` cell. It is replayed like every other program before anything is written.
* The code is short on purpose: rules as code, load, one gift, check, write.


In [ ]:
WRITE_SUBMISSION = False  # set True after Copy & Edit to write /kaggle/working/submission.csv


## Rules as code

In [ ]:
import csv, glob, json, os
csv.field_size_limit(1 << 30)
DATASET = 'santa-2026-elf-programs-and-experiments'
def find(name):
    hits = sorted(glob.glob(f'/kaggle/input/**/{name}', recursive=True), key=lambda h: (DATASET not in h, h))
    if not hits: raise FileNotFoundError(f'{name}: attach the dataset {DATASET}')
    return hits[0]

def load_puzzles(path):
    return {r['id']: dict(start=json.loads(r['starting_values']), targets=json.loads(r['targets']),
                          limit=int(r['value_limit'])) for r in csv.DictReader(open(path))}

def replay(p, prog):
    """Strict replay under the competition rules; returns the length or raises ValueError.
    48 registers, read registers must hold a value, values stay positive integers <= value_limit,
    subtraction needs a > b, division must be exact, every target must appear in some register."""
    R = [None] * 48
    for k, v in enumerate(p['start']): R[k] = v
    seen = set(p['start'])
    for op, i, j, d in prog:
        a, b = R[i], R[j]
        if op not in '+-*/' or a is None or b is None: raise ValueError('bad instruction')
        if op == '-' and a <= b: raise ValueError('non-positive subtraction')
        if op == '/' and a % b: raise ValueError('inexact division')
        v = a + b if op == '+' else a - b if op == '-' else a * b if op == '*' else a // b
        if not 0 < v <= p['limit']: raise ValueError('value out of range')
        R[d] = v; seen.add(v)
    if not set(p['targets']) <= seen: raise ValueError('missing target')
    return len(prog)

## Base: the dataset's best public programs

In [ ]:
P = load_puzzles(find('puzzles.csv'))
programs = {r['id']: json.loads(r['program']) for r in csv.DictReader(open(find('best_programs.csv')))}
BASE = sum(replay(P[k], programs[k]) for k in P)
print(f'base from the dataset: {BASE:,} instructions, all 1,000 programs replay correctly')

## 🎁 Behind today's door: the one change of this version

In [ ]:
# scored_0943: 88 -> 87 instructions (built from scratch by our own constructor)
CHANGE = {"scored_0943": [["*",0,1,4],["*",4,2,5],["+",2,5,6],["+",1,6,7],["+",0,4,8],["-",8,2,9],["*",6,9,10],["*",8,0,11],["*",2,5,12],["*",3,2,13],["+",11,3,14],["+",1,13,15],["*",15,2,16],["+",8,15,17],["*",17,3,18],["-",10,12,19],["+",19,15,20],["-",8,3,20],["*",2,7,21],["+",3,21,21],["-",21,5,22],["+",10,22,22],["-",21,13,23],["*",2,23,24],["-",24,22,24],["-",23,4,23],["-",23,12,24],["-",24,4,25],["-",25,3,26],["-",23,17,17],["+",17,6,23],["*",14,24,24],["-",21,18,18],["/",10,1,1],["+",7,22,21],["-",21,18,22],["+",22,20,24],["-",24,10,10],["+",10,5,24],["-",12,10,27],["*",24,0,0],["+",12,0,28],["+",28,8,28],["*",28,13,13],["-",28,25,29],["-",22,1,1],["+",1,21,30],["-",30,7,7],["+",7,11,31],["*",9,31,9],["+",16,9,9],["-",30,10,9],["-",21,11,10],["-",21,2,11],["-",11,0,0],["+",24,0,0],["+",23,12,11],["+",11,27,12],["*",0,15,15],["+",17,15,16],["*",4,0,0],["-",12,26,4],["*",26,4,16],["-",15,19,15],["*",20,15,16],["+",28,16,16],["+",7,0,0],["+",13,9,0],["-",29,23,0],["*",12,20,7],["+",7,0,0],["-",4,3,0],["+",31,0,0],["*",27,5,3],["-",0,14,0],["+",0,3,0],["+",1,11,0],["*",0,14,0],["-",15,18,1],["+",1,0,0],["-",9,2,0],["*",26,0,0],["-",0,25,0],["+",6,8,0],["*",0,31,0],["-",10,22,1],["-",0,1,0]]}

for pid, prog in CHANGE.items():
    old, new = replay(P[pid], programs[pid]), replay(P[pid], prog)   # the new program is checked like every other one
    assert new < old, (pid, old, new)
    programs[pid] = prog
    print(f'{pid}: {old} -> {new}')
TOTAL = sum(replay(P[k], programs[k]) for k in P)
print(f'total after the change: {TOTAL:,} instructions ({TOTAL - BASE:+d})')

## Write

In [ ]:
if WRITE_SUBMISSION:
    with open('/kaggle/working/submission.csv', 'w', newline='') as f:
        w = csv.writer(f); w.writerow(['id', 'program'])
        for pid in sorted(programs): w.writerow([pid, json.dumps(programs[pid], separators=(',', ':'))])
    print(f'submission.csv written: {TOTAL:,} instructions')
else:
    print('WRITE_SUBMISSION is False: no submission file in this output. Copy & Edit and set it to True.')